In [1]:
from the_well.data.normalization import ZScoreNormalization
from pathlib import Path
import numpy as np
import pickle
import fsspec
import torch
import yaml
import sys
import re

PROJECT_ROOT = "/Users/gabrieleaccarino/Workspace/waveletflux"
sys.path.insert(0, PROJECT_ROOT)

from waveletflux.utils.well_data import get_well_dataset, get_well_inputs
from waveletflux.modules.wavelet_fm import WaveletFlowModule

print("Imports OK | MPS:", torch.backends.mps.is_available())
print(f"MPS allocated : {torch.mps.current_allocated_memory() / 1e9:.2f} GB")
print(f"MPS reserved  : {torch.mps.driver_allocated_memory()  / 1e9:.2f} GB")

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

Imports OK | MPS: True
MPS allocated : 0.00 GB
MPS reserved  : 0.00 GB


In [2]:
#CONFIG_PATH = f"{PROJECT_ROOT}/configs/turbulent_radiative_layer_2D/WaveletFlow_turbulent_radiative_layer_2D_3scales_db2_v0.0.toml"
well_base_path    = f"{PROJECT_ROOT}/data/datasets"
#well_dataset_name = 'turbulent_radiative_layer_2D'
#well_dataset_name = 'rayleigh_benard'
well_dataset_name = 'planetswe'
stats_path        = f"{well_base_path}/{well_dataset_name}/stats.yaml"
#ckpt_name         = "WaveletFlow_turbulent_radiative_layer_2D_3scales_db2_v1.0_20260323_093126-epoch=188-val_loss=0.0217_portable.ckpt"
#ckpt_name         = "WaveletFlow_turbulent_radiative_layer_2D_1scales_db2_v1.0_20260323_102317-epoch=178-val_loss=0.0096_portable.ckpt"
#ckpt_name         = "WaveletFlow_turbulent_radiative_layer_2D_3scales_db2_last_v1.1_portable.ckpt"
#ckpt_name         = "WaveletFlow_turbulent_radiative_layer_2D_3scales_db2_v2.0_20260325_164613-epoch=192-val_loss=0.0147_portable.ckpt" 
#ckpt_name         = "WaveletFlow_turbulent_radiative_layer_2D_3scales_db2_v3.0_20260326_084546-epoch=253-val_loss=0.0183_portable.ckpt"
#ckpt_name         = "WaveletFlow_turbulent_radiative_layer_2D_3scales_db2_last_v3.0_20260326_084546-epoch=253-val_loss=0.0183_portable.ckpt"
#ckpt_name         = f"WaveletFlow_{well_dataset_name}_3scales_db2_v1.0_20260401_150336-epoch=194-val_loss=0.0207_portable.ckpt"
ckpt_name        = f"WaveletFlow_{well_dataset_name}_3scales_db2_v1.0_20260403_124641-epoch=192-val_loss=0.0023_portable.ckpt"
ckpt_base_name    = re.split(r"_\d{8}_", ckpt_name)[0]
ckpt_path         = f"{PROJECT_ROOT}/experiments/{well_dataset_name}/ckpts/{ckpt_name}"

In [3]:
with fsspec.open(stats_path, mode='r') as f:
    stats = yaml.safe_load(f)

model = WaveletFlowModule.from_checkpoint(ckpt_path, map_location=device)
model.eval()
model.to(device)
#model.rollout = torch.compile(model.rollout, backend="aot_eager")
print("Model loaded from checkpoint.")

/Users/gabrieleaccarino/Workspace/waveletflux/waveletflux/modules/wavelet_fm.py:743: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location=map_l

Model loaded from checkpoint.


In [4]:
dataset = get_well_dataset(
    path            = well_base_path,
    physics         = well_dataset_name,
    split           = 'test',
    steps           = -1,
    transform       = ['log_scalars'],   
    augment         = [],
)

In [7]:
N_TRAJ             = np.arange(len(dataset.metadata.n_steps_per_trajectory)) 
N_ROLLOUT_STEPS    = dataset.metadata.n_steps_per_trajectory[0]                         # autoregressive steps after the seed window
N_STEPS_INPUT      = 4                                                                  # Ti — sliding-window width (must match training)
NUM_SAMPLING_STEPS = 5                                                                 # ODE steps per prediction
N_ENSEMBLE         = 1                                                                 # flow-matching ensemble size
#device_ids         = [device_idx for device_idx in range(torch.cuda.device_count())]    # GPU device indices for DataParallel

In [8]:
"""pred_trajs = []
real_trajs = []

for traj_idx in N_TRAJ:
    print(f"\n{'━'*60}")
    print(f"  Trajectory {traj_idx}")
    print(f"{'━'*60}")

    item          = dataset[traj_idx]
    x_full, label = get_well_inputs(item, device=device)      # (N_ROLLOUT_STEPS, H, W, C), physical

    preds = model.rollout(
        seed          = x_full[:N_STEPS_INPUT].unsqueeze(0),  # (1, N_STEPS_INPUT, H, W, C), physical
        scalar_conds  = label.unsqueeze(0),                   # (1, num_cond), scalar conditions
        rollout_steps = N_ROLLOUT_STEPS,
        n_ensemble    = N_ENSEMBLE,
        verbose       = True,
    )
    # preds: (1, N_ENSEMBLE, N_ROLLOUT_STEPS, H, W, C)

    pred_trajs.append(preds.detach().cpu())
    real_trajs.append(x_full.unsqueeze(0).detach().cpu())"""

pred_trajs = []
real_trajs = []

with torch.inference_mode():
    for traj_idx in N_TRAJ:
        print(f"\n{'━'*60}")
        print(f"  Trajectory {traj_idx}")
        print(f"{'━'*60}")

        item          = dataset[traj_idx]
        x_full, label = get_well_inputs(item, device=device)  # (T, H, W, C), (num_cond,)

        seed = x_full[:N_STEPS_INPUT]                  # (N_STEPS_INPUT, H, W, C)

        # Expand along batch dim — each copy gets different noise during sampling
        seed_exp  = seed.unsqueeze(0).expand(N_ENSEMBLE, -1, -1, -1, -1)  # (N_ENS, T, H, W, C)
        
        if label is not None:
            label_exp = label.unsqueeze(0).expand(N_ENSEMBLE, -1)             # (N_ENS, num_cond)
        else:
            label_exp = None
            
        preds = model.rollout(
            seed          = seed_exp,    # (N_ENSEMBLE, T, H, W, C)
            scalar_conds  = label_exp,   # (N_ENSEMBLE, num_cond)
            rollout_steps = N_ROLLOUT_STEPS,
            n_ensemble    = 1,           # 1 member per batch element — diversity comes from batch noise
            verbose       = True,
        )
        # preds: (N_ENSEMBLE, 1, N_ROLLOUT_STEPS, H, W, C)
        preds = preds.squeeze(1)        # → (N_ENSEMBLE, N_ROLLOUT_STEPS, H, W, C)

        pred_trajs.append(preds.detach().unsqueeze(0).cpu())          # (1, N_ENS, T, H, W, C)
        real_trajs.append(x_full.unsqueeze(0).detach().cpu())


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 0
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  rollout call  201/201 (frames 201–201/201)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  rollout call  201/201 (frames 201–201/201)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  rollout call  201/201 (frames 201–201/201)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 3
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  rollout call  201/201 (frames 201–201/201)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 4
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  rollout call  201/201 (frames 201–201/201)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Trajectory 5
━━━━━━━━

# Save to binary

In [12]:
dataset[0].keys()

dict_keys(['input_fields', 'boundary_conditions', 'space_grid', 'input_time_grid'])

In [30]:
2.0031 - 1.0043

0.9987999999999999

In [51]:
dataset[9]['input_time_grid'].shape

torch.Size([201])

In [49]:
dataset[0]['input_time_grid'].diff()

tensor([1.0043, 0.9988, 0.9988, 1.0032, 1.0020, 1.0020, 0.9919, 1.0026, 1.0026,
        1.0026, 0.9924, 1.0026, 1.0026, 1.0025, 0.9924, 1.0026, 1.0026, 1.0025,
        0.9924, 1.0026, 1.0025, 1.0026, 0.9924, 1.0026, 1.0025, 1.0026, 0.9924,
        1.0026, 1.0025, 1.0016, 0.9995, 1.0001, 0.9937, 1.0049, 0.9937, 1.0049,
        1.0049, 0.9937, 1.0049, 0.9937, 1.0049, 0.9937, 1.0049, 1.0049, 0.9937,
        1.0049, 0.9937, 1.0049, 0.9937, 1.0049, 0.9937, 1.0049, 1.0049, 0.9937,
        1.0049, 0.9937, 1.0049, 0.9937, 1.0049, 1.0049, 0.9937, 1.0049, 0.9937,
        1.0049, 0.9937, 1.0049, 1.0049, 0.9987, 0.9965, 0.9965, 1.0082, 0.9965,
        0.9965, 0.9965, 1.0082, 0.9965, 0.9965, 1.0082, 0.9965, 0.9965, 1.0082,
        0.9965, 0.9965, 0.9965, 1.0082, 0.9965, 0.9965, 1.0082, 0.9965, 0.9965,
        1.0082, 0.9965, 0.9965, 0.9965, 1.0082, 0.9965, 0.9965, 1.0082, 0.9965,
        0.9965, 0.9965, 1.0082, 0.9965, 0.9965, 1.0082, 0.9965, 0.9965, 1.0082,
        0.9965, 0.9965, 1.0018, 0.9971, 

In [34]:
dataset[0].keys()

dict_keys(['input_fields', 'boundary_conditions', 'space_grid', 'input_time_grid'])

In [10]:
# save predicted trajectories to disk
save_dir = Path(f'{PROJECT_ROOT}/experiments/{well_dataset_name}/results')
save_dir.mkdir(exist_ok=True)

pred_trajs_stacked = torch.cat(pred_trajs, dim=0)  # (N_TRAJ, N_ENSEMBLE, N_ROLLOUT_STEPS, H, W, C)
save_path = save_dir / f"{ckpt_base_name}_rollout_nsampling_steps{NUM_SAMPLING_STEPS}_nensemble{N_ENSEMBLE}_traj.pkl"

with open(save_path, "wb") as f:
    pickle.dump(pred_trajs_stacked, f)

# save real trajectories to disk, save only one time!!
#real_trajs_stacked = torch.cat(real_trajs, dim=0)  # (N_TRAJ, N_ROLLOUT_STEPS, H, W, C)
#save_path = save_dir / f"ground_truth_{well_dataset_name}_traj.pkl"

#with open(save_path, "wb") as f:
#    pickle.dump(real_trajs_stacked, f)